# Delivery Time Prediction

**Regression Project 60**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict food delivery time based on order and delivery partner attributes: distance,
delivery person age, ratings, weather, traffic, vehicle type, city type, and order time.

This project simulates a real-world food delivery optimization problem where accurate
ETA prediction directly impacts customer satisfaction and operational efficiency.

## 3. Learning Objectives

1. Build a regression pipeline for logistics / delivery optimization
2. Handle mixed feature types (numeric, categorical, datetime)
3. Parse and engineer features from messy real-world columns
4. Understand delivery time drivers (distance, traffic, weather)
5. Evaluate with R², RMSE, MAE and residual analysis
6. Compare LazyPredict, FLAML, and gradient boosting models

## 4. Problem Statement

> Given order attributes and delivery conditions, predict the **delivery time** (in minutes).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Customers | Accurate ETA improves satisfaction |
| Delivery partners | Better route/time planning |
| Operations | Optimize driver allocation |
| Business | Reduce late deliveries and complaints |
| ML learners | Real-world messy data with geospatial features |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `gauravmalik26/food-delivery-dataset` |
| Records | ~45,593 |
| Features | Delivery_person_Age, Delivery_person_Ratings, Restaurant_latitude/longitude, Delivery_location_latitude/longitude, Weather, Road_traffic_density, Vehicle_condition, Type_of_order, Type_of_vehicle, City, etc. |
| Target | `Time_taken(min)` |
| Key insight | Distance and traffic are strongest predictors |

## 7. Dataset Source and License Notes

- **Kaggle**: [gauravmalik26/food-delivery-dataset](https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset)
- Simulated food delivery data
- License: check Kaggle dataset page
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'gauravmalik26/food-delivery-dataset'
TARGET = 'Time_taken(min)'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 120
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the food delivery dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
# Prefer train file
train_files = [f for f in csv_files if 'train' in os.path.basename(f).lower()]
chosen = train_files[0] if train_files else sorted(csv_files, key=os.path.getsize, reverse=True)[0]
df_raw = pd.read_csv(chosen)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Clean the target column (may have extra text) and validate data quality.

In [ ]:
df = df_raw.copy()
# The target column may have format like 'Time_taken(min)' with extra spaces or '(min) XX'
# Identify target
target_candidates = [c for c in df.columns if 'time' in c.lower() and 'taken' in c.lower()]
if target_candidates:
    TARGET = target_candidates[0]
elif TARGET not in df.columns:
    TARGET = df.columns[-1]

# Clean target: extract numeric value
if df[TARGET].dtype == object:
    df[TARGET] = df[TARGET].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')

print(f'Missing values (top):\n{df.isnull().sum()[df.isnull().sum() > 0].head(10)}\n')
print(f'Duplicated rows: {df.duplicated().sum()}')
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Target range: {df[TARGET].min():.0f} to {df[TARGET].max():.0f} min')

## 13. Exploratory Data Analysis

Explore delivery time drivers: distance, traffic, weather, vehicle, city type.

In [ ]:
df.describe()

In [ ]:
# Clean numeric columns
for col in df.columns:
    if df[col].dtype == object:
        # Strip whitespace
        df[col] = df[col].str.strip()
        # Try converting columns that should be numeric
        if col.lower() in ['delivery_person_age', 'delivery_person_ratings', 'multiple_deliveries']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
cat_feats = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_feats[:6]:
    print(f'{col}: {df[col].nunique()} unique values')

In [ ]:
# Delivery time by weather and traffic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
weather_col = [c for c in df.columns if 'weather' in c.lower()]
traffic_col = [c for c in df.columns if 'traffic' in c.lower()]
if weather_col:
    df.groupby(weather_col[0])[TARGET].mean().sort_values().plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title(f'Mean delivery time by {weather_col[0]}')
if traffic_col:
    df.groupby(traffic_col[0])[TARGET].mean().sort_values().plot.barh(ax=axes[1], color='coral')
    axes[1].set_title(f'Mean delivery time by {traffic_col[0]}')
plt.tight_layout(); plt.show()

In [ ]:
num_cols_eda = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols_eda) >= 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = df[num_cols_eda].corr()
    sns.heatmap(corr.iloc[:min(12,len(corr)),:min(12,len(corr))], annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout(); plt.show()

## 14. Target Analysis

Delivery times typically range from 10 to 50 minutes with some outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=40, color='coral', alpha=0.7)
axes[0].set_title('Delivery Time Distribution (min)')
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title('Delivery Time Box Plot')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():.1f} min')
print(f'Median: {df[TARGET].median():.1f} min')
print(f'Std: {df[TARGET].std():.1f} min')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split. First we compute distance and clean features.

In [ ]:
# Compute Haversine distance between restaurant and delivery location
def haversine(lat1, lon1, lat2, lon2):
    """Haversine distance in km."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

lat_cols = [c for c in df.columns if 'latitude' in c.lower()]
lon_cols = [c for c in df.columns if 'longitude' in c.lower()]

# Convert lat/lon to numeric
for c in lat_cols + lon_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

if len(lat_cols) >= 2 and len(lon_cols) >= 2:
    df['distance_km'] = haversine(
        df[lat_cols[0]], df[lon_cols[0]],
        df[lat_cols[1]], df[lon_cols[1]]
    )
    print(f'Distance computed. Mean: {df["distance_km"].mean():.2f} km')

# Drop raw lat/lon and ID columns
drop_cols = lat_cols + lon_cols
id_cols = [c for c in df.columns if c.lower() in ['id', 'delivery_person_id', 'restaurant_latitude',
           'restaurant_longitude', 'delivery_location_latitude', 'delivery_location_longitude']]
drop_cols = list(set(drop_cols + id_cols))
# Also drop order date/time columns — we'll extract features first
date_cols = [c for c in df.columns if 'date' in c.lower() or 'order' in c.lower() and 'type' not in c.lower()]

# Extract hour from time columns
time_cols = [c for c in df.columns if 'time' in c.lower() and c != TARGET and 'taken' not in c.lower()]
for tc in time_cols:
    try:
        parsed = pd.to_datetime(df[tc], errors='coerce')
        if parsed.notna().sum() > len(df) * 0.5:
            df[f'{tc}_hour'] = parsed.dt.hour
    except: pass

drop_cols = drop_cols + date_cols + time_cols
drop_cols = [c for c in drop_cols if c in df.columns and c != TARGET]
df = df.drop(columns=list(set(drop_cols)))

df = df.dropna(subset=[TARGET]).reset_index(drop=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric**: distance, age, ratings, multiple_deliveries → impute + scale
- **Categorical**: weather, traffic, vehicle, city, order type → OHE

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

- **Distance** already computed via Haversine
- **is_peak_hour**: Whether order was placed during lunch (11–14) or dinner (18–21)
- **is_experienced**: Whether delivery person has high ratings (≥4.5)

In [ ]:
def engineer_features(d):
    d = d.copy()
    hour_cols = [c for c in d.columns if '_hour' in c.lower()]
    if hour_cols:
        h = d[hour_cols[0]].fillna(12)
        d['is_peak_hour'] = (((h >= 11) & (h <= 14)) | ((h >= 18) & (h <= 21))).astype(int)
    rating_cols = [c for c in d.columns if 'rating' in c.lower()]
    if rating_cols:
        d['is_experienced'] = (pd.to_numeric(d[rating_cols[0]], errors='coerce') >= 4.5).astype(int)
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features: {X_train.shape[1]}')

## 18. Baseline Model

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:.1f} min, MAE={r['MAE']:.1f} min")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:.1f} min")

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:.1f} min, MAE={final[name]['MAE']:.1f} min")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.2, s=8)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual (min)'); ax.set_ylabel('Predicted (min)')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.2, s=8)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():.1f} min')
print(f'Median absolute error: {np.median(abs_errors):.1f} min')
print(f'90th percentile error: {np.percentile(abs_errors, 90):.1f} min')
print(f'Max error: {abs_errors.max():.1f} min')

## 24. Interpretation and Business Insight

- **Distance** (Haversine) is the single strongest predictor of delivery time
- **Road traffic density** significantly increases delivery time in jam/high-traffic areas
- **Weather conditions** (storms, sandstorms) add delay
- **City type** (Urban/Semi-Urban/Metropolitan) affects road conditions and distances
- **Vehicle type** (motorcycle vs scooter vs electric) has a moderate effect
- **Delivery person rating** is a proxy for experience and speed

## 25. Limitations

1. Simulated/synthetic data — real delivery times have more noise
2. No real-time traffic data integration
3. No restaurant preparation time
4. No order complexity (number of items, special instructions)
5. GPS coordinates may be imprecise

## 26. How to Improve This Project

1. Add real-time traffic API data
2. Include restaurant preparation time
3. Road network distance (not just straight-line Haversine)
4. Time-of-day and day-of-week interaction features
5. Ensemble stacking of top models

## 27. Production Considerations

- Real-time ETA display in customer app
- Dynamic driver assignment based on predicted time
- Traffic-aware route optimization
- Model retraining with new delivery data weekly
- A/B testing predicted vs actual ETA accuracy

## 28. Common Mistakes

1. Not computing distance from lat/lon coordinates
2. Keeping raw lat/lon as features (high precision noise)
3. Not parsing the target column (may have 'min' text)
4. Ignoring traffic density as a key feature
5. Not handling NaN values in ratings and age columns

## 29. Mini Challenge / Exercises

1. Try using road-network distance (e.g., via OSRM API) instead of Haversine
2. Build separate models for Urban vs Semi-Urban cities
3. Create a 'rush hour' feature and test its effect
4. Try quantile regression to predict delivery time ranges
5. Add weather severity scoring (clear=0, fog=1, storm=2) as ordinal feature

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict food delivery time |
| Dataset | Food Delivery (~45K records) |
| Difficulty | Medium |
| Key insight | Distance and traffic dominate delivery time |
| Best approach | Gradient boosting with Haversine distance |
| Primary metric | R² |
| Baseline comparison | Distance feature alone gives major R² lift |